In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## FAB (Fine-grained Adaptive Bidding)

Overview:
----------
This section loads advertiser-specific data files stored in Parquet format.
Each file corresponds to one advertiser, and the advertiser ID is extracted 
from the filename (assuming a format like 'advertiser_id.parquet').

Steps:
------
1. Define the path where the Parquet files are stored.
2. Use glob to list all files in the directory matching "*.parquet".
3. Iterate over the files:
    - Extract the advertiser ID from the filename.
    - Read the Parquet file into a Pandas DataFrame.
    - Store the DataFrame in a dictionary keyed by the advertiser ID.
4. Print the total number of advertisers loaded.

In [1]:
import pandas as pd
import numpy as np
import os
import glob

DATA_PATH = "path/to/advertiser/parquet/files/"

# Load all advertiser-specific files
advertiser_files = glob.glob(os.path.join(DATA_PATH, "*.parquet"))

# Create a dictionary to hold each advertiser's dataset
advertiser_data = {}
for file in advertiser_files:
    # Extract advertiser ID from filename (assuming filename format is advertiser_id.parquet)
    advertiser_id = os.path.basename(file).split('.')[0]
    advertiser_data[advertiser_id] = pd.read_parquet(file)

print(f"Loaded data for {len(advertiser_data)} advertisers.")

Overview:
----------
In this section, we prepare the data to compute key features required for the FAB strategy.
These features help the agent make informed bidding decisions and include:
  - pCTR: Predicted Click-Through Rate per time block.
  - win_rate: Auction win rate per time block.
  - avg_market_price: Average bidding price, used as a proxy for market competition.
  - budget_spent: Total cost spent per time block.
  - remaining_budget_ratio: The ratio of budget left after spending in a time block.

Steps:
------
1. Copy the DataFrame to avoid modifying the original data.
2. Group the data by 'time_block' and compute:
    - pCTR as the mean of 'click_bool'.
    - win_rate as the mean of 'impression_bool'.
    - avg_market_price as the mean of 'bidding_price'.
    - budget_spent as the sum of 'paying_price'.
3. Compute the remaining_budget_ratio as a proxy by subtracting the proportion of the budget spent.
4. Select the final features required by the RL model, including the unique 'bidId', 'advertiser_id', etc.

In [2]:
def preprocess_data(df):
    """
    Prepare data for FAB strategy by computing state features.
    
    Features computed:
      - pCTR: Predicted CTR per time block.
      - win_rate: Auction win rate per time block.
      - avg_market_price: Average bidding price (proxy for market competition).
      - budget_spent: Total cost (paying_price) spent per time block.
      - remaining_budget_ratio: Proxy using the sum of budget spent.
    """
    df = df.copy()

    # Compute CTR per time block
    df['pCTR'] = df.groupby('time_block')['click_bool'].transform('mean')
    
    # Compute Win Rate per time block
    df['win_rate'] = df.groupby('time_block')['impression_bool'].transform('mean')
    
    # Compute Market Competition Proxy (Avg Bidding Price)
    df['avg_market_price'] = df.groupby('time_block')['bidding_price'].transform('mean')

    # Compute Budget Spent and Remaining Budget Ratio
    df['budget_spent'] = df.groupby('time_block')['paying_price'].transform('sum')
    df['remaining_budget_ratio'] = 1 - (df['budget_spent'] / df['budget_spent'].sum())

    # Select final features needed for the RL model
    state_features = ['remaining_budget_ratio', 'budget_spent', 'pCTR', 'win_rate', 'avg_market_price']
    df = df[['bidId', 'advertiser_id', 'time_block', 'bidding_price'] + state_features]
    
    return df

Overview:
----------
This section implements the TD3 (Twin Delayed Deep Deterministic Policy Gradient) agent,
a state-of-the-art algorithm for continuous action spaces. The agent consists of:

1. **Actor Network:** Maps the state to an action (bidding factor). The output is bounded using Tanh.
2. **Critic Networks:** Two critics estimate the Q-value for a state-action pair. Having two helps mitigate 
   overestimation bias.
3. **TD3Agent Class:** Combines the actor and critics, including target networks for stable training,
   methods for selecting actions (with exploration noise), training from a replay buffer, and soft updates.

Key Points:
-----------
- **Exploration:** Noise is added to the actions during training for exploration.
- **Soft Update:** Target networks are updated slowly using the parameter `tau` for smoother learning.
- **Replay Buffer:** Experiences are stored and sampled in mini-batches for training.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Actor network: maps state to an action (bidding factor)
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Actor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()  # Output bounded between -1 and 1
        )

    def forward(self, state):
        return self.model(state)

# Critic network: estimates the Q-value for a given (state, action) pair
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim + action_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state, action):
        return self.model(torch.cat([state, action], dim=1))

# TD3 Agent: Uses twin critics and an actor to learn an optimal bidding policy
class TD3Agent:
    def __init__(self, state_dim, action_dim, lr=3e-4):
        self.actor = Actor(state_dim, action_dim)
        self.critic1 = Critic(state_dim, action_dim)
        self.critic2 = Critic(state_dim, action_dim)
        
        # Create target networks
        self.target_actor = Actor(state_dim, action_dim)
        self.target_critic1 = Critic(state_dim, action_dim)
        self.target_critic2 = Critic(state_dim, action_dim)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic1_optimizer = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_optimizer = optim.Adam(self.critic2.parameters(), lr=lr)

        # Copy initial weights to target networks
        self.target_actor.load_state_dict(self.actor.state_dict())
        self.target_critic1.load_state_dict(self.critic1.state_dict())
        self.target_critic2.load_state_dict(self.critic2.state_dict())

        self.gamma = 0.99
        self.tau = 0.005

    def select_action(self, state, noise=0.1):
        """
        Select an action based on the current state.
        Adds exploration noise to the action.
        """
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action = self.actor(state_tensor).detach().numpy()[0]
        return np.clip(action + np.random.normal(0, noise), -0.99, 0.99)

    def train(self, replay_buffer, batch_size=64):
        """
        Train the actor and critic networks using transitions sampled from the replay buffer.
        """
        if len(replay_buffer) < batch_size:
            return
        
        batch = random.sample(replay_buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states)
        actions = torch.FloatTensor(actions)
        rewards = torch.FloatTensor(rewards).unsqueeze(1)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones).unsqueeze(1)

        # Compute target actions with added noise
        target_actions = self.target_actor(next_states) + torch.clamp(torch.randn_like(actions) * 0.2, -0.5, 0.5)
        target_q1 = self.target_critic1(next_states, target_actions)
        target_q2 = self.target_critic2(next_states, target_actions)
        target_q = rewards + self.gamma * torch.min(target_q1, target_q2) * (1 - dones)

        # Calculate critic losses
        current_q1 = self.critic1(states, actions)
        current_q2 = self.critic2(states, actions)
        critic1_loss = nn.MSELoss()(current_q1, target_q.detach())
        critic2_loss = nn.MSELoss()(current_q2, target_q.detach())

        # Update the critics
        self.critic1_optimizer.zero_grad()
        critic1_loss.backward()
        self.critic1_optimizer.step()

        self.critic2_optimizer.zero_grad()
        critic2_loss.backward()
        self.critic2_optimizer.step()

        # Compute and update the actor loss
        actor_loss = -self.critic1(states, self.actor(states)).mean()
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Soft update target networks
        for target, source in zip(self.target_actor.parameters(), self.actor.parameters()):
            target.data.copy_(self.tau * source.data + (1 - self.tau) * target.data)

        for target, source in zip(self.target_critic1.parameters(), self.critic1.parameters()):
            target.data.copy_(self.tau * source.data + (1 - self.tau) * target.data)

        for target, source in zip(self.target_critic2.parameters(), self.critic2.parameters()):
            target.data.copy_(self.tau * source.data + (1 - self.tau) * target.data)

Overview:
----------
This section defines a custom Gym environment that simulates the Real-Time Bidding (RTB) process.
In this environment:
  - **State:** A vector containing metrics such as remaining budget ratio, cost ratio, CTR, win rate, and average market price.
  - **Action:** A continuous bidding factor that adjusts the final bid price.
  - **Reward:** Calculated based on the number of clicks achieved and the cost incurred (normalized by the total budget).

Steps in Each Step:
-------------------
1. For the current time block, filter the data.
2. Compute the final bid price as a function of the pCTR and a modified market price (adjusted by the bidding factor).
3. Simulate the auction:
    - Mark auctions as 'won' if the final bid exceeds the average market price.
    - Compute cost only for won auctions.
4. Compute the reward as the sum of pCTR (clicks) for won auctions minus the normalized cost.
5. Update the remaining budget and move to the next time block.
6. Define the next state using updated metrics.
7. Determine if the episode (campaign) is finished (either all time blocks are done or budget is exhausted).


In [ ]:
import gym
from gym import spaces

class RTBEnv(gym.Env):
    """
    Custom Environment for Real-Time Bidding (RTB).
    
    The environment simulates auction rounds (time blocks) where the agent
    selects a bidding factor (action) to determine the final bid price.
    The state includes budget and performance metrics.
    """
    def __init__(self, df, budget=1000000):
        super(RTBEnv, self).__init__()
        self.df = df
        self.total_budget = budget
        self.remaining_budget = budget
        self.time_blocks = df['time_block'].unique()
        self.current_time_block = 0

        # State space: [remaining_budget_ratio, cost_ratio, CTR, win_rate, avg_market_price]
        self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)
        # Action space: Continuous bidding factor in the range (-0.99, 0.99)
        self.action_space = spaces.Box(low=-0.99, high=0.99, shape=(1,), dtype=np.float32)
    
    def step(self, action):
        """
        Execute one time-block step in the bidding process.
        
        - Computes the final bid based on the pCTR and bidding factor.
        - Simulates the auction outcome.
        - Calculates reward based on clicks (pCTR sum) and cost efficiency.
        """
        bidding_factor = action[0]

        # Filter the data for the current time block
        block_data = self.df[self.df['time_block'] == self.time_blocks[self.current_time_block]]

        # Compute final bid price
        block_data['final_bid'] = block_data['pCTR'] * (block_data['avg_market_price'] / (1 + bidding_factor))

        # Simulate auction outcome: won if final_bid exceeds avg_market_price
        block_data['won'] = block_data['final_bid'] > block_data['avg_market_price']
        # Calculate cost spent if won
        block_data['spent'] = block_data['won'] * block_data['bidding_price']

        # Compute reward: sum of pCTR for won auctions minus normalized cost
        clicks = block_data[block_data['won']]['pCTR'].sum()
        cost = block_data['spent'].sum()
        reward = clicks - (cost / self.total_budget)

        # Update the remaining budget and move to the next time block
        self.remaining_budget -= cost
        self.current_time_block += 1

        # Compute the next state features
        remaining_budget_ratio = self.remaining_budget / self.total_budget
        cost_ratio = cost / self.total_budget
        win_rate = block_data['won'].mean()
        ctr = block_data['pCTR'].mean()
        avg_market_price = block_data['avg_market_price'].mean()

        next_state = np.array([remaining_budget_ratio, cost_ratio, ctr, win_rate, avg_market_price], dtype=np.float32)
        done = self.current_time_block >= len(self.time_blocks) or self.remaining_budget <= 0
        
        return next_state, reward, done, {}

    def reset(self):
        """
        Reset the environment to the initial state at the beginning of a new campaign.
        """
        self.remaining_budget = self.total_budget
        self.current_time_block = 0
        return np.array([1, 0, self.df['pCTR'].mean(), self.df['win_rate'].mean(), self.df['avg_market_price'].mean()], dtype=np.float32)


Overview:
----------
For each advertiser, we train the FAB strategy using the TD3 agent within our custom RTB environment.

Training Process:
-----------------
1. **Environment Initialization:** Create an instance of the RTBEnv using the preprocessed data.
2. **Agent Initialization:** Instantiate the TD3 agent.
3. **Replay Buffer:** Use a deque to store transitions (state, action, reward, next_state, done) for experience replay.
4. **Episode Loop:** For a given number of episodes:
    - Reset the environment.
    - For each time block:
        - The agent selects an action based on the current state.
        - The environment executes the action, returning the next state, reward, and done flag.
        - Store the transition in the replay buffer.
    - After processing time blocks, train the agent using mini-batches from the replay buffer.
    - Print periodic updates (every 50 episodes) showing the episode reward.
5. **Return the Trained Agent:** The final trained model for that advertiser is returned.

In [ ]:
from collections import deque
import random

def train_fab_for_advertiser(advertiser_id, df, episodes=500):
    """
    Train the FAB (Fine-grained Adaptive Bidding) strategy for a single advertiser.
    
    Parameters:
      - advertiser_id: Identifier for the advertiser.
      - df: Preprocessed data for the advertiser.
      - episodes: Number of training episodes.
    
    Returns:
      - Trained TD3Agent model.
    """
    env = RTBEnv(df)
    agent = TD3Agent(state_dim=5, action_dim=1)
    replay_buffer = deque(maxlen=10000)
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        
        # Iterate over all time blocks for the episode
        for _ in range(len(env.time_blocks)):
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)
            
            # Store the transition in the replay buffer
            replay_buffer.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward
            
            if done:
                break
        
        # Train the agent using mini-batches from the replay buffer
        agent.train(replay_buffer, batch_size=64)
        
        if episode % 50 == 0:
            print(f"Advertiser {advertiser_id} - Episode {episode}: Reward = {total_reward}")

    return agent


Overview:
----------
This final section iterates over each advertiser's dataset, performs preprocessing, and then trains the 
FAB model using the TD3 agent in the custom RTB environment.

Steps:
------
1. Loop through the dictionary containing advertiser data.
2. For each advertiser:
    - Print the advertiser ID for logging.
    - Preprocess the data to compute required features.
    - Train the FAB model for that advertiser using the training loop defined earlier.
3. Store the trained model in a dictionary, keyed by the advertiser ID.

In [ ]:
trained_models = {}

for advertiser_id, df in advertiser_data.items():
    print(f"Training FAB for Advertiser: {advertiser_id}")
    processed_df = preprocess_data(df)
    trained_models[advertiser_id] = train_fab_for_advertiser(advertiser_id, processed_df, episodes=500)

## Modified FAB

Overview:
---------
FAB (Fine-grained Adaptive Bidding) is superior to classical neural networks for RTB
for three key reasons:

1️⃣ **Sequential Decision-Making vs. Static Prediction:**
   - *Classical NN:* Predict bid prices from historical data without real-time adaptation.
   - *FAB (RL-based):* Dynamically adjusts bids based on current auction conditions,
     leading to higher efficiency and better budget utilization.

2️⃣ **Lower Computational Overhead at Inference:**
   - *Classical NN:* Requires heavy matrix operations (10–50ms per bid).
   - *FAB:* Computes a single action (bidding factor λ) per time slot,
     reducing inference time to sub-millisecond levels.

3️⃣ **Adaptive Budget Allocation Across Time Slots:**
   - *Classical NN:* Treats each impression independently, often leading to suboptimal budget use.
   - *FAB:* Uses RL to strategically distribute budget over time, ensuring optimal ad placements.

**Modifications for Ultra-Fast Inference (<5ms):**

To further reduce inference latency while maintaining performance, we propose:
1️⃣ **Lightweight Policy Approximation Model:**
   - Replace complex RL (e.g., TD3) with a simple linear model that approximates bidding factors.
2️⃣ **Precomputed Bidding Factors:**
   - Precompute optimal bidding factors for each time slot and store them in a lookup table.
3️⃣ **Quantized Models / Edge Optimizations:**
   - Use model quantization or deploy on optimized inference engines (e.g., ONNX Runtime, TensorRT).

In this modified FAB, we simulate ultra-fast inference by using a simple linear model
(as a lightweight policy) with precomputed weights.

In [9]:
import pandas as pd
import numpy as np
import os
import glob

# Path where your advertiser Parquet files are stored
DATA_PATH = "path/to/advertiser/parquet/files/"

# Load all advertiser-specific files from the DATA_PATH
advertiser_files = glob.glob(os.path.join(DATA_PATH, "*.parquet"))

# Create a dictionary to hold each advertiser's dataset
advertiser_data = {}
for file in advertiser_files:
    # Extract advertiser ID from filename (assumes filename format is advertiser_id.parquet)
    advertiser_id = os.path.basename(file).split('.')[0]
    advertiser_data[advertiser_id] = pd.read_parquet(file)

print(f"Loaded data for {len(advertiser_data)} advertisers.")

Loaded data for 0 advertisers.


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [10]:
import numpy as np

In [11]:
import pandas as pd
import os

# Path to the single advertiser Parquet file
file_path = "/kaggle/input/devcraft-lesgo/ipinyou/1458.parquet"

# Extract advertiser ID from filename (assuming filename format: advertiser_id.parquet)
advertiser_id = os.path.basename(file_path).split('.')[0]

# Load the Parquet file into a DataFrame and store it in a dictionary
advertiser_data = {advertiser_id: pd.read_parquet(file_path)}

print(f"Loaded data for advertiser {advertiser_id}.")


Loaded data for advertiser 1458.


## 2. Data Preprocessing

Preprocessing computes key features required for the FAB strategy:
  - pCTR: Predicted Click-Through Rate per time block.
  - win_rate: Auction win rate per time block.
  - avg_market_price: Average bidding price (proxy for market competition).
  - budget_spent: Total cost spent per time block.
  - remaining_budget_ratio: Proxy for remaining budget.

In [12]:
def preprocess_data(df):
    df = df.copy()

    # Compute pCTR per time block: average click probability
    df['pCTR'] = df.groupby('time_block')['click_bool'].transform('mean')
    
    # Compute win rate per time block: average impression win rate
    df['win_rate'] = df.groupby('time_block')['impression_bool'].transform('mean')
    
    # Compute market competition proxy: average bidding price
    df['avg_market_price'] = df.groupby('time_block')['bidding_price'].transform('mean')

    # Compute budget spent and remaining budget ratio per time block
    df['budget_spent'] = df.groupby('time_block')['paying_price'].transform('sum')
    df['remaining_budget_ratio'] = 1 - (df['budget_spent'] / df['budget_spent'].sum())

    # Select final features for the model
    state_features = ['remaining_budget_ratio', 'budget_spent', 'pCTR', 'win_rate', 'avg_market_price']
    df = df[['bidId', 'advertiser_id', 'time_block', 'bidding_price'] + state_features]
    
    return df


Instead of a complex RL agent (e.g., TD3), we use a lightweight policy approximation.
This simple linear model computes the bidding factor (λ) as a fast dot-product
with precomputed weights, ensuring inference time is well below 5ms.


In [13]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import PolynomialFeatures

class PolyRegressor:
    def __init__(self, degree, random_state=None):
        self.poly = PolynomialFeatures(degree)
        self.reg = SGDRegressor(max_iter=1000, tol=1e-3, random_state=random_state)
        self.fitted = False

    def partial_fit(self, X, y):
        if not self.fitted:
            # Fit the polynomial transformer and the regressor the first time
            X_poly = self.poly.fit_transform(X)
            self.reg.partial_fit(X_poly, y)
            self.fitted = True
        else:
            X_poly = self.poly.transform(X)
            self.reg.partial_fit(X_poly, y)

    def predict(self, X):
        X_poly = self.poly.transform(X)
        return self.reg.predict(X_poly)


In [14]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

class EnsembleRegressor:
    def __init__(self, input_dim, degree=2, random_state=None):
        # Linear regressor
        self.linear = SGDRegressor(max_iter=1000, tol=1e-3, random_state=random_state)
        # Use our custom polynomial regressor instead of a pipeline.
        self.poly = PolyRegressor(degree, random_state=random_state)
        self.is_initialized = False

    def predict(self, X):
        """
        Predict by averaging the outputs of the linear and polynomial models.
        X: np.array of shape (n_samples, input_dim)
        """
        linear_pred = self.linear.predict(X)
        poly_pred = self.poly.predict(X)
        return (linear_pred + poly_pred) / 2.0

    def partial_fit(self, X, y):
        """
        Perform an online update using partial_fit.
        """
        if not self.is_initialized:
            self.linear.partial_fit(X, y)
            self.poly.partial_fit(X, y)
            self.is_initialized = True
        else:
            self.linear.partial_fit(X, y)
            self.poly.partial_fit(X, y)



class EnsemblePolicy:
    """
    Actor network replacement using an ensemble of regressors.
    Maps a state to a bidding factor.
    """
    def __init__(self, state_dim, random_state=None):
        self.model = EnsembleRegressor(input_dim=state_dim, degree=2, random_state=random_state)
        # Initialize the model with dummy data to avoid NotFittedError.
        self.model.partial_fit(np.zeros((1, state_dim)), np.array([0]))
    
    def select_action(self, state):
        X = np.array(state).reshape(1, -1)
        action = self.model.predict(X)[0]
        return np.array([np.clip(action, -0.99, 0.99)], dtype=np.float32)
    
    def update(self, state, target_action):
        X = np.array(state).reshape(1, -1)
        y = np.array(target_action).reshape(1)
        self.model.partial_fit(X, y)


class EnsembleCritic:
    """
    Critic network replacement using an ensemble of regressors.
    Estimates Q-value given a state-action pair.
    """
    def __init__(self, state_dim, action_dim, random_state=None):
        # Input dimension is the concatenation of state and action.
        self.model = EnsembleRegressor(input_dim=state_dim + action_dim, degree=2, random_state=random_state)
        # Initialize with dummy data to avoid NotFittedError.
        self.model.partial_fit(np.zeros((1, state_dim + action_dim)), np.array([0]))
    
    def predict(self, state, action):
        X = np.concatenate([state, action]).reshape(1, -1)
        return self.model.predict(X)[0]
    
    def update(self, state, action, target_q):
        X = np.concatenate([state, action]).reshape(1, -1)
        y = np.array([target_q])
        self.model.partial_fit(X, y)

In [15]:
import copy
class LightweightFABAgent:
    """
    Agent with an ensemble-based policy (actor) and twin ensemble-based critics.
    Mimics TD3 but replaces neural networks with regressors.
    """
    def __init__(self, state_dim, action_dim, gamma=0.99, tau=0.005, random_state=None):
        self.actor = EnsemblePolicy(state_dim, random_state=random_state)
        self.critic1 = EnsembleCritic(state_dim, action_dim, random_state=random_state)
        self.critic2 = EnsembleCritic(state_dim, action_dim, random_state=random_state)
        # Initialize target networks as deep copies.
        self.target_actor = copy.deepcopy(self.actor)
        self.target_critic1 = copy.deepcopy(self.critic1)
        self.target_critic2 = copy.deepcopy(self.critic2)
        self.gamma = gamma
        self.tau = tau
    
    def select_action(self, state):
        return self.actor.select_action(state)
    
    def update_target_networks(self, hard_update=False):
        """
        For simplicity, we perform a hard update (copying online parameters).
        """
        self.target_actor = copy.deepcopy(self.actor)
        self.target_critic1 = copy.deepcopy(self.critic1)
        self.target_critic2 = copy.deepcopy(self.critic2)
    
    def train(self, replay_buffer, batch_size=64):
        if len(replay_buffer) < batch_size:
            return
        
        batch = random.sample(replay_buffer, batch_size)
        for state, action, reward, next_state, done in batch:
            # Compute target action using the target actor.
            target_action = self.target_actor.select_action(next_state)
            # Evaluate target Q-values from both target critics.
            q1_next = self.target_critic1.predict(next_state, target_action)
            q2_next = self.target_critic2.predict(next_state, target_action)
            target_q = reward + self.gamma * (0 if done else min(q1_next, q2_next))
            
            # Update both critics with the target Q-value.
            self.critic1.update(state, action, target_q)
            self.critic2.update(state, action, target_q)
            
            # Actor update: we wish to choose an action that maximizes critic1’s Q-value.
            # Here, we use a grid search over possible actions (since action space is 1D).
            possible_actions = np.linspace(-0.99, 0.99, num=21)
            best_action = None
            best_q = -np.inf
            for a in possible_actions:
                a_val = np.array([a], dtype=np.float32)
                q_val = self.critic1.predict(state, a_val)
                if q_val > best_q:
                    best_q = q_val
                    best_action = a_val
            # Update actor with the best action found.
            self.actor.update(state, best_action)
        
        # Update target networks (hard update for simplicity)
        self.update_target_networks(hard_update=True)


In [16]:
import gym
from gym import spaces
class RTBEnv(gym.Env):
    """
    Custom Gym environment for Real-Time Bidding (RTB).
    - State: [remaining_budget_ratio, cost_ratio, CTR, win_rate, avg_market_price]
    - Action: Bidding factor (λ) in range (-0.99, 0.99)
    - Reward: Based on clicks (sum of pCTR for won auctions) minus normalized cost.
    """
    def __init__(self, df, budget=1000000):
        super(RTBEnv, self).__init__()
        self.df = df
        self.total_budget = budget
        self.remaining_budget = budget
        self.time_blocks = df['time_block'].unique()
        self.current_time_block = 0

        self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)
        self.action_space = spaces.Box(low=-0.99, high=0.99, shape=(1,), dtype=np.float32)
    
    def step(self, action):
        bidding_factor = action[0]
        block_data = self.df[self.df['time_block'] == self.time_blocks[self.current_time_block]].copy()

        # Compute final bid price: pCTR * (avg_market_price / (1 + bidding_factor))
        block_data['final_bid'] = block_data['pCTR'] * (block_data['avg_market_price'] / (1 + bidding_factor))
        block_data['won'] = block_data['final_bid'] > block_data['avg_market_price']
        block_data['spent'] = block_data['won'] * block_data['bidding_price']

        clicks = block_data[block_data['won']]['pCTR'].sum()
        cost = block_data['spent'].sum()
        reward = clicks - (cost / self.total_budget)

        self.remaining_budget -= cost
        self.current_time_block += 1

        remaining_budget_ratio = self.remaining_budget / self.total_budget
        cost_ratio = cost / self.total_budget
        win_rate = block_data['won'].mean() if not block_data['won'].empty else 0
        ctr = block_data['pCTR'].mean()
        avg_market_price = block_data['avg_market_price'].mean()

        next_state = np.array([remaining_budget_ratio, cost_ratio, ctr, win_rate, avg_market_price], dtype=np.float32)
        done = self.current_time_block >= len(self.time_blocks) or self.remaining_budget <= 0
        
        return next_state, reward, done, {}

    def reset(self):
        self.remaining_budget = self.total_budget
        self.current_time_block = 0
        return np.array([
            1, 
            0, 
            self.df['pCTR'].mean(), 
            self.df['win_rate'].mean(), 
            self.df['avg_market_price'].mean()
        ], dtype=np.float32)

In [17]:
from collections import deque
import random

def train_modified_fab_for_advertiser(advertiser_id, df, episodes=50):
    env = RTBEnv(df)
    agent = LightweightFABAgent(state_dim=5, action_dim=1, random_state=42)
    replay_buffer = deque(maxlen=10000)
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        
        # Run through all time blocks in the campaign
        for _ in range(len(env.time_blocks)):
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)
            replay_buffer.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward
            if done:
                break
        
        agent.train(replay_buffer, batch_size=32)
        if episode % 10 == 0:
            print(f"Advertiser {advertiser_id} - Episode {episode}: Total Reward = {total_reward}")
    
    return agent

In [19]:
modified_trained_models = {}

for advertiser_id, df in advertiser_data.items():
    print(f"Running Modified Ultra-Fast FAB for Advertiser: {advertiser_id}")
    processed_df = preprocess_data(df)
    agent = train_modified_fab_for_advertiser(advertiser_id, processed_df, episodes=50)
    modified_trained_models[advertiser_id] = agent

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Running Modified Ultra-Fast FAB for Advertiser: 1458
Advertiser 1458 - Episode 0: Total Reward = 0.0
Advertiser 1458 - Episode 10: Total Reward = 0.0
Advertiser 1458 - Episode 20: Total Reward = 0.0
Advertiser 1458 - Episode 30: Total Reward = 0.0
Advertiser 1458 - Episode 40: Total Reward = 0.0
